In [2]:
#————————————确认文件能正常读取————————————————————————————————

# 1. 导入需要的工具（复制运行即可）
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 2. 读取你的节点表（含age、era列）
df_node = pd.read_csv("带时代标签的节点表 (1).csv", encoding="utf-8-sig")
# 读取下载的中心性表
df_centrality = pd.read_csv("marvel_node_centrality.csv", encoding="utf-8-sig")

# 3. 检查是否读成功（看前2行数据）
print("✅ 节点表前2行（含age列）：")
print(df_node[["Id", "Label", "age", "era"]].head(2))  # 只看关键列，避免信息太多
print("\n✅ 中心性表前2行（含中心性指标）：")
print(df_centrality.head(2))

✅ 节点表前2行（含age列）：
                       Id                   Label   age         era
0  Black Panther / T'chal  Black Panther / T'chal    35  Silver Age
1        Loki [asgardian]        Loki [asgardian]  1000  Silver Age

✅ 中心性表前2行（含中心性指标）：
                       Id                   Label gender   age  \
0  Black Panther / T'chal  Black Panther / T'chal   male    35   
1        Loki [asgardian]        Loki [asgardian]   Male  1000   

                        team（复仇者联盟、银河护卫队、神盾局）  \
0  Avengers, Wakandan Royal Guard, Illuminati   
1    Acts of Vengeance, Cabal, Young Avengers   

  team(正派 (Hero)、反派 (Villain)、中立 (Neutral) -主属性-只写英文）   race  \
0                                               Hero   Black   
1                                            Villain   White   

  first_appearance（剧中） cleaned_year         era  Degree Centrality  \
0                 1966         1966  Silver Age           0.309816   
1                 1962         1962  Silver Age           0.190184   

   Betwe

In [4]:
import pandas as pd
import numpy as np

# 1. 读取中心性表
df = pd.read_csv("marvel_node_centrality.csv", encoding="utf-8-sig")

# 2. 查看age列情况（为了确认数据类型）
print("❓ age列原始数据类型：", df['age'].dtype)
print("❓ 查看前10个age值：", df['age'].head(10).tolist())

# 3. 核心清洗函数（修复版）
# 功能：提取数字+5 (20s→25, 30s→35)，空值/异常值返回NaN
def clean_age_final(age_val):
    # 如果本身是NaN或空值，直接返回
    if pd.isna(age_val):
        return np.nan
        
    # 转换成字符串处理（强制转换）
    age_str = str(age_val)
    
    # 情况A：包含 "s" (如 20s, 30s, 40s) -> 提取数字+5
    if "s" in age_str:
        # 提取字符串中的所有数字字符
        digits = ''.join([c for c in age_str if c.isdigit()])
        if digits: # 如果成功提取到数字
            return float(digits) + 5.0
        else:
            return np.nan # 提取不到，标记为空
    
    # 情况B：纯数字/其他格式 (如 35, 1000) -> 直接转浮点数
    try:
        return float(age_val)
    except:
        return np.nan # 转换失败，标记为空

# 4. 应用清洗函数
df["cleaned_age"] = df["age"].apply(clean_age_final)

# 5. 验证结果（这一步不报错就代表成功了）
print("\n✅ 清洗完成！结果预览：")
# 对比原始age和清洗后的cleaned_age
print(df[["age", "cleaned_age"]].head(10))

# 6. 统计清洗效果
print(f"\n📊 统计信息：")
print(f"原始数据总量：{len(df)}")
print(f"成功清洗的有效年龄数据：{df['cleaned_age'].notna().sum()} 条")
print(f"年龄范围：{df['cleaned_age'].min():.1f} ~ {df['cleaned_age'].max():.1f}")

❓ age列原始数据类型： str
❓ 查看前10个age值： ['35', '1000', '30', '25', '25', '28', '45', '27', '10', '120']

✅ 清洗完成！结果预览：
    age  cleaned_age
0    35         35.0
1  1000       1000.0
2    30         30.0
3    25         25.0
4    25         25.0
5    28         28.0
6    45         45.0
7    27         27.0
8    10         10.0
9   120        120.0

📊 统计信息：
原始数据总量：327
成功清洗的有效年龄数据：265 条
年龄范围：5.0 ~ 10000.0


In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------------------- 1. 读取中心性表 ----------------------
df = pd.read_csv("marvel_node_centrality.csv", encoding="utf-8-sig")
print("✅ 读取成功：共有", len(df), "个节点")
print("\n📋 数据表列名核对：", df.columns.tolist())

# ---------------------- 2. 年龄清洗（20s→25 / 30s→35） ----------------------
def clean_age_final(age_val):
    if pd.isna(age_val):
        return np.nan
    
    age_str = str(age_val)
    # 处理带 "s" 的格式（20s/30s/40s）
    if "s" in age_str:
        digits = ''.join([c for c in age_str if c.isdigit()])
        return float(digits) + 5.0 if digits else np.nan
    # 处理普通数字格式
    try:
        return float(age_val)
    except:
        return np.nan

df["cleaned_age"] = df["age"].apply(clean_age_final)
print("\n✅ 年龄清洗完成")
print("清洗前age示例：", df["age"].head(5).tolist())
print("清洗后cleaned_age示例：", df["cleaned_age"].head(5).tolist())

# ---------------------- 3. 过滤可分析数据（仅保留有效年龄） ----------------------
# ---------------------- 3. 过滤可分析数据（仅保留0-100岁的有效年龄角色） ----------------------
# 新增：(df["cleaned_age"] >= 0) & (df["cleaned_age"] <= 100) 这部分，限制年龄范围
df_analysis = df[
    (df["cleaned_age"] >= 0) & (df["cleaned_age"] <= 100)  # 只留0-100岁
].dropna(subset=["cleaned_age"]).copy()  # 删掉空值
print("\n✅ 过滤后可分析数据量（0-100岁）：", len(df_analysis))  # 会显示筛选后的角色数，比如200+
# ---------------------- 4. 计算相关性矩阵 ----------------------
corr_cols = [
    "cleaned_age",
    "Degree Centrality",
    "Betweenness Centrality",
    "Closeness Centrality",
    "Eigenvector Centrality",
    "Clustering Coefficient"
]
corr_matrix = df_analysis[corr_cols].corr()
age_corr = corr_matrix["cleaned_age"].drop("cleaned_age")

print("\n📊 年龄与各中心性的相关系数：")
print(age_corr.round(4))

# ---------------------- 5. 英文可视化（无中文依赖，无乱码） ----------------------
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
# 左图：年龄 vs 度中心性
ax1.scatter(df_analysis["cleaned_age"], df_analysis["Degree Centrality"], alpha=0.6, s=30, color="#2E86AB")
ax1.set_xlabel("Age (Years)")
ax1.set_ylabel("Degree Centrality")
ax1.set_title(f"Age vs Degree Centrality\nCorr: {age_corr['Degree Centrality']:.4f}")
ax1.grid(alpha=0.3)

# 右图：相关性热力图
simplified_labels = ["Age", "Degree", "Betweenness", "Closeness", "Eigenvector", "Clustering"]
im = ax2.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
ax2.set_xticks(range(len(corr_cols)))
ax2.set_yticks(range(len(corr_cols)))
ax2.set_xticklabels(simplified_labels, rotation=45, ha="right")
ax2.set_yticklabels(simplified_labels)
ax2.set_title("Correlation Heatmap")

# 标注热力图数值
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax2.text(j, i, f"{corr_matrix.iloc[i,j]:.4f}", ha="center", va="center", color="black")

plt.tight_layout()
plt.savefig("Age_Centrality_Correlation.png", dpi=300, bbox_inches="tight")
plt.close()
print("\n✅ 可视化图已保存：Age_Centrality_Correlation.png")

# ---------------------- 6. 输出最终交付文件（修复列名错误） ----------------------
# 1）全量分析数据表（仅保留100%存在的列，彻底避免KeyError）
df_analysis[
    ["Id", "Label", "cleaned_age", "era", "gender",
     "Degree Centrality", "Betweenness Centrality", "Closeness Centrality"]
].to_csv("Age_Centrality_Analysis_Data.csv", index=False, encoding="utf-8-sig")

# 2）相关性结果汇总表（汇报直接用）
corr_summary = pd.DataFrame({
    "Centrality_Metric": age_corr.index,
    "Correlation_with_Age": age_corr.values.round(4),
    "Strength": [
        "Strong" if abs(x)>=0.5 else
        "Moderate" if abs(x)>=0.3 else
        "Weak" if abs(x)>=0.1 else
        "None" for x in age_corr.values
    ]
})
corr_summary.to_csv("Age_Centrality_Correlation_Summary.csv", index=False, encoding="utf-8-sig")

print("\n🎉 全流程执行完毕！生成3个交付文件：")
print("1. Age_Centrality_Analysis_Data.csv → 全量分析原始数据")
print("2. Age_Centrality_Correlation_Summary.csv → 相关系数汇总表")
print("3. Age_Centrality_Correlation.png → 可直接插入PPT的可视化图表")

✅ 读取成功：共有 327 个节点

📋 数据表列名核对： ['Id', 'Label', 'gender', 'age', 'team（复仇者联盟、银河护卫队、神盾局）', 'team(正派 (Hero)、反派 (Villain)、中立 (Neutral) -主属性-只写英文）', 'race', 'first_appearance（剧中）', 'cleaned_year', 'era', 'Degree Centrality', 'Betweenness Centrality', 'Closeness Centrality', 'Eigenvector Centrality', 'Clustering Coefficient']

✅ 年龄清洗完成
清洗前age示例： ['35', '1000', '30', '25', '25']
清洗后cleaned_age示例： [35.0, 1000.0, 30.0, 25.0, 25.0]

✅ 过滤后可分析数据量（0-100岁）： 246

📊 年龄与各中心性的相关系数：
Degree Centrality        -0.1072
Betweenness Centrality   -0.0603
Closeness Centrality     -0.1271
Eigenvector Centrality   -0.1397
Clustering Coefficient    0.0254
Name: cleaned_age, dtype: float64

✅ 可视化图已保存：Age_Centrality_Correlation.png

🎉 全流程执行完毕！生成3个交付文件：
1. Age_Centrality_Analysis_Data.csv → 全量分析原始数据
2. Age_Centrality_Correlation_Summary.csv → 相关系数汇总表
3. Age_Centrality_Correlation.png → 可直接插入PPT的可视化图表
